In [1]:
!date

Sat Jun  1 11:30:13 PDT 2024


In [2]:
%load_ext autoreload
%load_ext line_profiler

In [3]:
import logging
logging.basicConfig(level=logging.INFO, force=True)

In [4]:
import os as _os
_os.chdir(_os.environ['PROJECT_ROOT'])

In [6]:
import graph_tool as gt
import graph_tool.draw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
from contextlib import contextmanager
import xarray as xr
from itertools import product
from tqdm import tqdm
from itertools import chain
from graph_tool.util import find_edge
import scipy as sp

import strainzip as sz
from strainzip.pandas_util import idxwhere
import strainzip.app.deconvolve

In [25]:
sz.deconvolution.num_minimal_complete_pathsets(5, 5)

120

In [83]:
from itertools import permutations, combinations_with_replacement
from strainzip.deconvolution import PathSet, LocalPath, num_

# Assuming that M < N
# Pick one, arbitrary order of the longer list (presumably index order [1...N])
# All pathsets can be constructed by then taking the shorter list and:
# Take every permutation of the shorter list (M! of these)
# Then append every random sample of the shorter list of size |N-M|, with replacement. (M^(N-M) of these)
# Match this list of length max(N, M) with the longer list to construct all pairs.


def minimal_complete_pathsets(n, m):
    flipped = n < m
    if flipped:
        n, m = m, n
    fixed_order = range(n)
    for permutation_of_m in permutations(range(m), r=m):
        for remaining_sample in combinations_with_replacement(range(m), r=(n - m)):
            selected_order = list(permutation_of_m) + list(remaining_sample)
            if not flipped:
                yield PathSet(frozenset(LocalPath(*pair) for pair in zip(fixed_order, selected_order)), n=n, m=m)
            else:
                yield PathSet(frozenset(LocalPath(*pair) for pair in zip(selected_order, fixed_order)), n=m, m=n)
            
list(minimal_complete_pathsets(4, 4))

[PathSet(paths=frozenset({LocalPath(left=1, right=1), LocalPath(left=3, right=3), LocalPath(left=2, right=2), LocalPath(left=0, right=0)}), n=4, m=4),
 PathSet(paths=frozenset({LocalPath(left=2, right=3), LocalPath(left=1, right=1), LocalPath(left=3, right=2), LocalPath(left=0, right=0)}), n=4, m=4),
 PathSet(paths=frozenset({LocalPath(left=3, right=3), LocalPath(left=1, right=2), LocalPath(left=2, right=1), LocalPath(left=0, right=0)}), n=4, m=4),
 PathSet(paths=frozenset({LocalPath(left=2, right=3), LocalPath(left=1, right=2), LocalPath(left=3, right=1), LocalPath(left=0, right=0)}), n=4, m=4),
 PathSet(paths=frozenset({LocalPath(left=3, right=2), LocalPath(left=1, right=3), LocalPath(left=2, right=1), LocalPath(left=0, right=0)}), n=4, m=4),
 PathSet(paths=frozenset({LocalPath(left=3, right=1), LocalPath(left=2, right=2), LocalPath(left=1, right=3), LocalPath(left=0, right=0)}), n=4, m=4),
 PathSet(paths=frozenset({LocalPath(left=0, right=1), LocalPath(left=1, right=0), LocalPath(le

In [65]:

in_edges = {0, 1}
out_edges = {0, 1}

num_possible_paths = len(in_edges) * len(out_edges)
cardinality_complete_pathset = max(len(in_edges), len(out_edges))
max_in_partnerset_cardinality = max(len(in_edges) - len(out_edges), 0) + 1
max_out_partnerset_cardinality = max(len(out_edges) - len(in_edges), 0) + 1

cardinality_complete_pathset

2

In [59]:
[(p, pathset_to_design(p, 2, 2)) for p in PathSet(paths=frozenset([LocalPath(left=0, right=0)]), n=2, m=2).iter_neighbors()]

ValueError: need at least one array to stack

In [54]:

        


[pathset_to_design(p, 2, 2) for p in PathSet(paths=frozenset({LocalPath(left=0, right=0), LocalPath(left=1, right=1)}), n=2, m=2).iter_neighbors()]

# @cache
# def design_paths(n, m):
#     in_designs = np.eye(n, dtype=int)
#     out_designs = np.eye(m, dtype=int)
#     design_products = product(in_designs, out_designs)
#     left_and_right_product = product(range(n), range(m))
#     design = [np.concatenate([in_row, out_row]) for in_row, out_row in design_products]
#     return {LocalPath


# def formulate_path_deconvolution(in_flows, out_flows):
#     n, m = in_flows.shape[0], out_flows.shape[0]
#     s = in_flows.shape[1]
#     assert in_flows.shape[1] == out_flows.shape[1]
#     X, labels = design_paths(n, m)
#     y = np.concatenate([in_flows, out_flows], axis=0)
#     return X, y, labels



[array([[0],
        [1],
        [0],
        [1]]),
 array([[1],
        [0],
        [1],
        [0]]),
 array([[1, 0],
        [0, 1],
        [0, 1],
        [1, 0]]),
 array([[1, 1, 0],
        [0, 0, 1],
        [1, 0, 0],
        [0, 1, 1]]),
 array([[1, 0, 0],
        [0, 1, 1],
        [1, 1, 0],
        [0, 0, 1]])]